In [1]:
!pip install -qU langchain langchain-openai langchain-qdrant

In [2]:
import getpass #for input password interface
import os #for read and manage python directory path and environment
from google.colab import userdata

#input required credentials
if not os.environ.get("OPENAI_API_KEY"):
  try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
    os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
  except:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")
    os.environ["QDRANT_API_KEY"] = getpass.getpass("Enter API key for QDRANT_API_KEY: ")
    os.environ["QDRANT_URL"] = getpass.getpass("Enter url for QDRANT_URL: ")

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [5]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

#setup qdrant client
client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)
#get all collection in your qdrant vector database
collections_response = client.get_collections()
print("Collections:", collections_response.collections)

Collections: [CollectionDescription(name='product_documents')]


In [6]:
qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name="product_documents",
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    check_compatibility=False
)

In [11]:
from langchain.tools import tool
from datetime import datetime

@tool
def current_datetime():
  """Tool for retrieving the current date and time information."""
  return datetime.now().strftime("%A")

@tool
def promo_list():
  """Use this tool for get information about promo details per day."""
  promo = '''List of dicount everyday:
* Monday: Discount 50% for all product!
* Tuesday: Free shipping cost for total payment over Rs. 200.
* Wednesday: Buy 2 Get 1 Free for all product!
* Thrusday: Free shipping cost for total payment over Rs. 100.
* Friday: Discount 30% for all product!
* Saturday: Free shipping cost for total payment over Rs. 150.
* Sunday: Discount 20% for all product!
'''
  return promo

@tool
def check_relevant_products(query: str):
  """Tool for retrieving informations about relevant products in the store."""
  documents = qdrant.similarity_search(query, k=10)
  docs = [d.page_content for d in documents]
  return docs

In [12]:
from langchain.agents import create_agent


tools = [check_relevant_products, promo_list, current_datetime]
agent = create_agent(model, tools, system_prompt="You are a clothing store assistant, please politely answer the question. Always run current_datetime tool at first step. Use any tools for get the best answer.")

In [13]:
agent.invoke({"messages": [{"role": "user", "content": "what is today promo?"}]})

{'messages': [HumanMessage(content='what is today promo?', additional_kwargs={}, response_metadata={}, id='476fa718-a5fe-447f-a354-ea9c3a668ddc'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 124, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_83e2dd34fc', 'id': 'chatcmpl-DVHFXT0ELxYN6OFrs31eMjozKKpeF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9685-d727-7e32-b6f2-ea029360ef83-0', tool_calls=[{'name': 'current_datetime', 'args': {}, 'id': 'call_BiKefdACCYXGJkwk3NU6uS9h', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 10, 't

In [14]:
#build interactive chat, Type "exit" to stop interaction.
user_input = input("Type your message: ")
while user_input.lower() not in ["exit", "q"]:
  prompt = {"messages": [{"role": "user", "content": user_input}]}
  response = agent.invoke(prompt)
  print(response["messages"][-1].content)
  user_input = input("Type your message: ")

Type your message: Helloo
Hello! How can I assist you today?
Type your message: What is today promo?
Today's promotion is free shipping for total payments over Rs. 100. If you have any questions or need assistance with your order, feel free to ask!
Type your message: recommend me a suit for man
Here are some recommended men's suits and formal wear options available in our store:

1. **Lee Mark Men's Solid Formal Shirt**
   - Price: Rs. 599
   - Description: This stylish solid formal shirt features long sleeves, a point collar, and a full-buttoned chest placket. It pairs well with chinos and loafers for a sophisticated look.

2. **Peter England Men's Solid Formal Shirt**
   - Price: Rs. 1399
   - Description: This formal shirt in a refined green color is perfect for professional settings. It combines quality and style for a polished appearance.

3. **Fashion Tree Men's Solid Formal Shirt (Dark Blue)**
   - Price: Rs. 999
   - Description: Great for office wear or formal events, this dar